<a href="https://colab.research.google.com/github/Jimmyqst/Path-Loss-ML-Optimization/blob/main/XGBoost_(Entrenamiento_Validaci%C3%B3n-TRAIN_VALIDATION).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Entrenamiento y Validación UPLINK**

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ==============================================================================
# 0. ESTILO DE TESIS (CONFIGURACIÓN MAESTRA)
# ==============================================================================
def configurar_estilo_tesis():
    sns.set_theme(style="whitegrid", context="paper")
    plt.rcParams.update({
        'font.family': 'serif',
        'axes.labelsize': 11,
        'axes.titlesize': 12,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'figure.titlesize': 14,
        'grid.alpha': 0.3,
        'grid.linestyle': '--',
        'lines.linewidth': 2,
    })

# Color Oficial XGBoost (Verde)
COLOR_XGB = '#2ca02c'

configurar_estilo_tesis()

# ==============================================================================
# 1. CARGA DE DATOS (UPLINK)
# ==============================================================================
DIR_DATA = '/content/drive/MyDrive/TESIS 100%/Datos_Entrenamiento_Finales/UpLink/'
DIR_OUTPUT = '/content/drive/MyDrive/TESIS 100%/XGBoost/Uplink/Entrenamiento/'

if not os.path.exists(DIR_OUTPUT):
    os.makedirs(DIR_OUTPUT)

print("⚙️ INICIANDO ENTRENAMIENTO XGBOOST UPLINK (ESTILO TESIS)...")

try:
    train_df = pd.read_csv(os.path.join(DIR_DATA, 'Train_UpLink.csv'))
    val_df   = pd.read_csv(os.path.join(DIR_DATA, 'Val_UpLink.csv'))
    print(f"✅ Datos cargados. Train: {len(train_df)} | Val: {len(val_df)}")
except FileNotFoundError:
    print("❌ Error: Archivos no encontrados."); exit()

# Detección de columnas
posibles_sf = ['SpreedFactor', 'SpreedFactor_UpLink', 'SpreadingFactor', 'SF']
col_sf = next((c for c in posibles_sf if c in train_df.columns), None)
if not col_sf: print("❌ SF no encontrado"); exit()

FEATURES = ['Distance', 'Gateway_Altitude', 'EndDevice_Altitude', col_sf, 'Terreno', 'Power_UpLink']
TARGET = next((c for c in ['Path_Loss_UpLink', 'Path_Loss_Uplink'] if c in train_df.columns), None)

if not TARGET: print("❌ Target no encontrado"); exit()

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val, y_val     = val_df[FEATURES], val_df[TARGET]

# ==============================================================================
# 2. ENTRENAMIENTO XGBOOST
# ==============================================================================
xgb_model = XGBRegressor(
    n_estimators=1200, learning_rate=0.05, max_depth=6, subsample=0.8,
    colsample_bytree=0.8, n_jobs=-1, random_state=42,
    objective='reg:squarederror', eval_metric=['rmse', 'mae'],
    early_stopping_rounds=20
)

print("\n🔥 Entrenando XGBoost (esto puede tardar un poco)...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=0
)

# Recuperar historial
results = xgb_model.evals_result()
epochs = len(results['validation_0']['rmse'])
x_axis = range(0, epochs)

# Métricas base
train_rmse = results['validation_0']['rmse']
val_rmse   = results['validation_1']['rmse']
train_mae  = results['validation_0']['mae']
val_mae    = results['validation_1']['mae']
# Calculamos MSE al cuadrado
train_mse  = [x**2 for x in train_rmse]
val_mse    = [x**2 for x in val_rmse]

# Mejor iteración
best_iter = xgb_model.best_iteration
best_ntree = best_iter + 1

# --- CÁLCULO DE R2 (Optimizado por muestreo para no saturar) ---
print("📌 Generando curva de R2 (muestreo)...")
train_r2 = []
val_r2 = []
puntos_grafica = 50 # Aumentamos un poco la resolución
step = max(1, epochs // puntos_grafica)
axis_r2 = list(range(step, epochs, step))
if axis_r2[-1] != epochs: axis_r2.append(epochs)

for i in axis_r2:
    # iteration_range requiere enteros
    limite = min(i, best_ntree + 50) # No predecir mucho más allá del óptimo para ahorrar tiempo
    if limite > epochs: limite = epochs

    p_tr = xgb_model.predict(X_train, iteration_range=(0, i))
    p_va = xgb_model.predict(X_val, iteration_range=(0, i))
    train_r2.append(r2_score(y_train, p_tr))
    val_r2.append(r2_score(y_val, p_va))

# ==============================================================================
# 3. DASHBOARD 2x2 DE MÉTRICAS (XGBOOST)
# ==============================================================================
print("\n📊 Generando Dashboard de Métricas...")
fig, axs = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle(f'XGBoost Uplink: Curvas de Aprendizaje (Óptimo: {best_ntree} iteraciones)', fontweight='bold')

# Helper para plotear
def plot_metric(ax, x_data, y_train, y_val, title, ylabel, is_r2=False):
    ax.plot(x_data, y_train, '--', color='gray', label='Train', alpha=0.7)
    ax.plot(x_data, y_val, color=COLOR_XGB, label='Validación', linewidth=2.5)

    if is_r2:
        # Para R2 usamos el eje muestreado
        idx_best = min(range(len(x_data)), key=lambda i: abs(x_data[i] - best_ntree))
        val_opt = y_val[idx_best]
        x_opt = x_data[idx_best]
    else:
        # Para el resto usamos el índice directo
        if best_iter < len(y_val):
            val_opt = y_val[best_iter]
            x_opt = best_ntree
        else:
            val_opt = y_val[-1]
            x_opt = len(y_val)

    ax.axvline(x=best_ntree, color='black', linestyle=':', alpha=0.5)
    ax.scatter([x_opt], [val_opt], color='black', zorder=5)

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Iteraciones (Árboles)', fontweight='bold')
    ax.set_ylabel(ylabel, fontweight='bold')
    ax.legend(loc='best')

# Plotting
plot_metric(axs[0, 0], x_axis, train_rmse, val_rmse, 'RMSE', 'Error (dB)')
plot_metric(axs[0, 1], x_axis, train_mse, val_mse, 'MSE', 'Error Cuadrático (dB²)')
plot_metric(axs[1, 0], x_axis, train_mae, val_mae, 'MAE', 'Error Absoluto (dB)')
plot_metric(axs[1, 1], axis_r2, train_r2, val_r2, 'R²', 'Score (0-1)', is_r2=True)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(os.path.join(DIR_OUTPUT, '1_Dashboard_Metricas.png'), dpi=300)
plt.close()

# ==============================================================================
# 4. ANÁLISIS DE ERROR AVANZADO
# ==============================================================================
print("🔍 Generando Gráficas de Análisis...")
# Predicción final con el mejor modelo
p_val = xgb_model.predict(X_val, iteration_range=(0, best_ntree))
residuos = y_val - p_val

# --- A. Dispersión ---
plt.figure(figsize=(8, 8))
plt.scatter(y_val, p_val, alpha=0.5, color=COLOR_XGB, edgecolor='w', s=50)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], '--', lw=2, color='black', label='Ideal')
plt.title('XGBoost Uplink: Real vs Predicho', fontweight='bold')
plt.xlabel('Path Loss Real (dB)', fontweight='bold')
plt.ylabel('Path Loss Predicho (dB)', fontweight='bold')
plt.legend()
plt.savefig(os.path.join(DIR_OUTPUT, '2_Dispersion_Real_vs_Predicho.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- B. Histograma ---
plt.figure(figsize=(10, 6))
sns.histplot(residuos, kde=True, color=COLOR_XGB, bins=30, line_kws={'linewidth': 2})
plt.axvline(x=0, color='black', linestyle='--', linewidth=1.5)
plt.title(f'Distribución de Errores (Media: {residuos.mean():.2f} dB)', fontweight='bold')
plt.xlabel('Error (dB)', fontweight='bold'); plt.ylabel('Frecuencia', fontweight='bold')
plt.savefig(os.path.join(DIR_OUTPUT, '3_Distribucion_Errores.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- C. Homocedasticidad ---
plt.figure(figsize=(10, 6))
plt.scatter(p_val, residuos, alpha=0.5, color=COLOR_XGB, edgecolor='w')
plt.axhline(y=0, color='black', linestyle='--', lw=2)
plt.title('Homocedasticidad: Residuos vs Predicción', fontweight='bold')
plt.xlabel('Path Loss Predicho (dB)', fontweight='bold'); plt.ylabel('Residuo (dB)', fontweight='bold')
plt.savefig(os.path.join(DIR_OUTPUT, '4_Homocedasticidad.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- D. QQ-Plot ---
plt.figure(figsize=(10, 6))
stats.probplot(residuos, dist="norm", plot=plt)
plt.title('QQ-Plot: Normalidad de Residuos', fontweight='bold')
plt.xlabel('Cuantiles Teóricos', fontweight='bold'); plt.ylabel('Valores Ordenados', fontweight='bold')
plt.gca().get_lines()[0].set_markerfacecolor(COLOR_XGB)
plt.gca().get_lines()[0].set_markeredgecolor(COLOR_XGB)
plt.gca().get_lines()[1].set_color('black')
plt.grid(True, linestyle='--', alpha=0.3)
plt.savefig(os.path.join(DIR_OUTPUT, '5_QQ_Plot.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- E. CDF ---
sorted_abs_error = np.sort(np.abs(residuos))
p = 1. * np.arange(len(sorted_abs_error)) / (len(sorted_abs_error) - 1)
plt.figure(figsize=(10, 6))
plt.plot(sorted_abs_error, p, linewidth=3, color=COLOR_XGB)
plt.title('CDF: Probabilidad Error Absoluto', fontweight='bold')
plt.xlabel('Error Absoluto (dB)', fontweight='bold'); plt.ylabel('Probabilidad', fontweight='bold')
idx_90 = np.searchsorted(p, 0.90)
err_90 = sorted_abs_error[idx_90]
plt.axvline(x=err_90, color='black', linestyle=':')
plt.text(err_90, 0.5, f' 90% < {err_90:.2f} dB', fontweight='bold')
plt.savefig(os.path.join(DIR_OUTPUT, '6_CDF_Error.png'), dpi=300, bbox_inches='tight'); plt.close()

# ==============================================================================
# 5. TABLA DE RESULTADOS FINAL
# ==============================================================================
print("📋 Generando Tabla de Resultados...")

rmse_v = np.sqrt(mean_squared_error(y_val, p_val))
mse_v  = mean_squared_error(y_val, p_val)
mae_v  = mean_absolute_error(y_val, p_val)
r2_v   = r2_score(y_val, p_val)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis('tight'); ax.axis('off')

data_table = [
    ['Métrica (XGBoost Uplink)', 'Valor Óptimo'],
    ['Mejor Iteración', f"{best_ntree}"],
    ['RMSE', f"{rmse_v:.4f} dB"],
    ['MAE', f"{mae_v:.4f} dB"],
    ['MSE', f"{mse_v:.4f} dB²"],
    ['R² Score', f"{r2_v:.4f}"]
]

table = ax.table(cellText=data_table, colLabels=None, cellLoc='center', loc='center')
table.auto_set_font_size(False); table.set_fontsize(12); table.scale(1, 1.8)

# Estilo Verde
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor(COLOR_XGB) # Cabecera Verde
    else:
        cell.set_facecolor('#f8f9fa')
        cell.set_edgecolor('#dddddd')

plt.title('Resumen de Desempeño XGBoost (Uplink)', fontsize=14, fontweight='bold', y=0.95)
plt.savefig(os.path.join(DIR_OUTPUT, '7_Tabla_Metricas_Finales.png'), dpi=300, bbox_inches='tight')
plt.close()

# Guardar modelo
joblib.dump(xgb_model, os.path.join(DIR_OUTPUT, 'Modelo_XGBoost_Uplink.pkl'))

print(f"\n✅ Procesamiento terminado. Gráficas y Tabla guardadas en: {DIR_OUTPUT}")

# **Entrenamiento DOWNLINK**

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ==============================================================================
# 0. ESTILO DE TESIS (CONFIGURACIÓN MAESTRA)
# ==============================================================================
def configurar_estilo_tesis():
    sns.set_theme(style="whitegrid", context="paper")
    plt.rcParams.update({
        'font.family': 'serif',
        'axes.labelsize': 11,
        'axes.titlesize': 12,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'figure.titlesize': 14,
        'grid.alpha': 0.3,
        'grid.linestyle': '--',
        'lines.linewidth': 2,
    })

# Color Oficial XGBoost (Verde)
COLOR_XGB = '#2ca02c'

configurar_estilo_tesis()

# ==============================================================================
# 1. CARGA DE DATOS (DOWNLINK ROBUSTO)
# ==============================================================================
DIR_DATA = '/content/drive/MyDrive/TESIS 100%/Datos_Entrenamiento_Finales/DownLink/'
DIR_OUTPUT = '/content/drive/MyDrive/TESIS 100%/XGBoost/Downlink/Entrenamiento/'

if not os.path.exists(DIR_OUTPUT):
    os.makedirs(DIR_OUTPUT)

print("⚙️ INICIANDO ENTRENAMIENTO XGBOOST DOWNLINK (ESTILO TESIS)...")

def detect_column(df, candidates, name_for_error):
    col = next((c for c in candidates if c in df.columns), None)
    if col is None:
        print(f"❌ Falta columna {name_for_error}. Columnas: {list(df.columns)}")
        raise SystemExit
    return col

try:
    train_df = pd.read_csv(os.path.join(DIR_DATA, 'Train_DownLink.csv'))
    val_df   = pd.read_csv(os.path.join(DIR_DATA, 'Val_DownLink.csv'))
    print(f"✅ Datos cargados. Train: {len(train_df)} | Val: {len(val_df)}")
except Exception as e:
    print(f"❌ Error cargando CSVs: {e}"); exit()

# Detectar SF
sf_candidates = ['SpreedFactor', 'SpreedFactor_DownLink', 'SpreedFactor_UpLink', 'SpreadingFactor', 'SF']
col_sf = detect_column(train_df, sf_candidates, "Spreading Factor")
# Sincronizar nombre en Val
col_sf_val = next((c for c in sf_candidates if c in val_df.columns), None)
if col_sf_val and col_sf_val != col_sf:
    val_df.rename(columns={col_sf_val: col_sf}, inplace=True)

# Detectar Target
target_candidates = ['Path_Loss_DownLink', 'Path_Loss_Downlink', 'Path_Loss']
TARGET = detect_column(train_df, target_candidates, "Target Path Loss")
# Sincronizar nombre en Val
target_val = next((c for c in target_candidates if c in val_df.columns), None)
if target_val and target_val != TARGET:
    val_df.rename(columns={target_val: TARGET}, inplace=True)

FEATURES = ['Distance', 'Gateway_Altitude', 'EndDevice_Altitude', col_sf, 'Terreno', 'Power_DownLink']

# Verificación final
for c in FEATURES + [TARGET]:
    if c not in train_df.columns: print(f"❌ Falta en Train: {c}"); exit()
    if c not in val_df.columns: print(f"❌ Falta en Val: {c}"); exit()

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val, y_val     = val_df[FEATURES], val_df[TARGET]

# ==============================================================================
# 2. ENTRENAMIENTO XGBOOST
# ==============================================================================
xgb_model = XGBRegressor(
    n_estimators=1200, learning_rate=0.05, max_depth=6, subsample=0.8,
    colsample_bytree=0.8, n_jobs=-1, random_state=42,
    objective='reg:squarederror', eval_metric=['rmse', 'mae'],
    early_stopping_rounds=20
)

print("\n🔥 Entrenando XGBoost Downlink...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=0
)

# Recuperar historial
results = xgb_model.evals_result()
epochs = len(results['validation_0']['rmse'])
x_axis = range(0, epochs)

train_rmse = results['validation_0']['rmse']
val_rmse   = results['validation_1']['rmse']
train_mae  = results['validation_0']['mae']
val_mae    = results['validation_1']['mae']
train_mse  = [x**2 for x in train_rmse]
val_mse    = [x**2 for x in val_rmse]

# Mejor iteración
best_iter = xgb_model.best_iteration
best_ntree = best_iter + 1

# --- CÁLCULO DE R2 (OPTIMIZADO) ---
print("📌 Generando curva de R2 (muestreo)...")
train_r2 = []
val_r2 = []
puntos_grafica = 50
step = max(1, epochs // puntos_grafica)
axis_r2 = list(range(step, epochs, step))
if axis_r2[-1] != epochs: axis_r2.append(epochs)

for i in axis_r2:
    limite = min(i, best_ntree + 50)
    if limite > epochs: limite = epochs

    p_tr = xgb_model.predict(X_train, iteration_range=(0, i))
    p_va = xgb_model.predict(X_val, iteration_range=(0, i))
    train_r2.append(r2_score(y_train, p_tr))
    val_r2.append(r2_score(y_val, p_va))

# ==============================================================================
# 3. DASHBOARD 2x2 DE MÉTRICAS (XGBOOST DL)
# ==============================================================================
print("\n📊 Generando Dashboard de Métricas...")
fig, axs = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle(f'XGBoost Downlink: Curvas de Aprendizaje (Óptimo: {best_ntree} iteraciones)', fontweight='bold')

def plot_metric(ax, x_data, y_train, y_val, title, ylabel, is_r2=False):
    ax.plot(x_data, y_train, '--', color='gray', label='Train', alpha=0.7)
    ax.plot(x_data, y_val, color=COLOR_XGB, label='Validación', linewidth=2.5)

    if is_r2:
        idx_best = min(range(len(x_data)), key=lambda i: abs(x_data[i] - best_ntree))
        val_opt = y_val[idx_best]
        x_opt = x_data[idx_best]
    else:
        if best_iter < len(y_val):
            val_opt = y_val[best_iter]
            x_opt = best_ntree
        else:
            val_opt = y_val[-1]
            x_opt = len(y_val)

    ax.axvline(x=best_ntree, color='black', linestyle=':', alpha=0.5)
    ax.scatter([x_opt], [val_opt], color='black', zorder=5)

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Iteraciones (Árboles)', fontweight='bold')
    ax.set_ylabel(ylabel, fontweight='bold')
    ax.legend(loc='best')

plot_metric(axs[0, 0], x_axis, train_rmse, val_rmse, 'RMSE', 'Error (dB)')
plot_metric(axs[0, 1], x_axis, train_mse, val_mse, 'MSE', 'Error Cuadrático (dB²)')
plot_metric(axs[1, 0], x_axis, train_mae, val_mae, 'MAE', 'Error Absoluto (dB)')
plot_metric(axs[1, 1], axis_r2, train_r2, val_r2, 'R²', 'Score (0-1)', is_r2=True)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(os.path.join(DIR_OUTPUT, '1_Dashboard_Metricas_DL.png'), dpi=300)
plt.close()

# ==============================================================================
# 4. ANÁLISIS DE ERROR AVANZADO (DL)
# ==============================================================================
print("🔍 Generando Gráficas de Análisis...")
p_val = xgb_model.predict(X_val, iteration_range=(0, best_ntree))
residuos = y_val - p_val

# --- A. Dispersión ---
plt.figure(figsize=(8, 8))
plt.scatter(y_val, p_val, alpha=0.5, color=COLOR_XGB, edgecolor='w', s=50)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], '--', lw=2, color='black', label='Ideal')
plt.title('XGBoost DL: Real vs Predicho', fontweight='bold')
plt.xlabel('Path Loss Real (dB)', fontweight='bold')
plt.ylabel('Path Loss Predicho (dB)', fontweight='bold')
plt.legend()
plt.savefig(os.path.join(DIR_OUTPUT, '2_Dispersion_DL.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- B. Histograma ---
plt.figure(figsize=(10, 6))
sns.histplot(residuos, kde=True, color=COLOR_XGB, bins=30, line_kws={'linewidth': 2})
plt.axvline(x=0, color='black', linestyle='--', linewidth=1.5)
plt.title(f'Distribución de Errores (Media: {residuos.mean():.2f} dB)', fontweight='bold')
plt.xlabel('Error (dB)', fontweight='bold'); plt.ylabel('Frecuencia', fontweight='bold')
plt.savefig(os.path.join(DIR_OUTPUT, '3_Distribucion_Errores_DL.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- C. Homocedasticidad ---
plt.figure(figsize=(10, 6))
plt.scatter(p_val, residuos, alpha=0.5, color=COLOR_XGB, edgecolor='w')
plt.axhline(y=0, color='black', linestyle='--', lw=2)
plt.title('Homocedasticidad: Residuos vs Predicción', fontweight='bold')
plt.xlabel('Path Loss Predicho (dB)', fontweight='bold'); plt.ylabel('Residuo (dB)', fontweight='bold')
plt.savefig(os.path.join(DIR_OUTPUT, '4_Homocedasticidad_DL.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- D. QQ-Plot ---
plt.figure(figsize=(10, 6))
stats.probplot(residuos, dist="norm", plot=plt)
plt.title('QQ-Plot: Normalidad de Residuos', fontweight='bold')
plt.xlabel('Cuantiles Teóricos', fontweight='bold'); plt.ylabel('Valores Ordenados', fontweight='bold')
plt.gca().get_lines()[0].set_markerfacecolor(COLOR_XGB)
plt.gca().get_lines()[0].set_markeredgecolor(COLOR_XGB)
plt.gca().get_lines()[1].set_color('black')
plt.grid(True, linestyle='--', alpha=0.3)
plt.savefig(os.path.join(DIR_OUTPUT, '5_QQ_Plot_DL.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- E. CDF ---
sorted_abs_error = np.sort(np.abs(residuos))
p = 1. * np.arange(len(sorted_abs_error)) / (len(sorted_abs_error) - 1)
plt.figure(figsize=(10, 6))
plt.plot(sorted_abs_error, p, linewidth=3, color=COLOR_XGB)
plt.title('CDF: Probabilidad Error Absoluto', fontweight='bold')
plt.xlabel('Error Absoluto (dB)', fontweight='bold'); plt.ylabel('Probabilidad', fontweight='bold')
idx_90 = np.searchsorted(p, 0.90)
err_90 = sorted_abs_error[idx_90]
plt.axvline(x=err_90, color='black', linestyle=':')
plt.text(err_90, 0.5, f' 90% < {err_90:.2f} dB', fontweight='bold')
plt.savefig(os.path.join(DIR_OUTPUT, '6_CDF_Error_DL.png'), dpi=300, bbox_inches='tight'); plt.close()

# ==============================================================================
# 5. TABLA DE RESULTADOS FINAL (DL)
# ==============================================================================
print("📋 Generando Tabla de Resultados...")

rmse_v = np.sqrt(mean_squared_error(y_val, p_val))
mse_v  = mean_squared_error(y_val, p_val)
mae_v  = mean_absolute_error(y_val, p_val)
r2_v   = r2_score(y_val, p_val)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis('tight'); ax.axis('off')

data_table = [
    ['Métrica (XGBoost Downlink)', 'Valor Óptimo'],
    ['Mejor Iteración', f"{best_ntree}"],
    ['RMSE', f"{rmse_v:.4f} dB"],
    ['MAE', f"{mae_v:.4f} dB"],
    ['MSE', f"{mse_v:.4f} dB²"],
    ['R² Score', f"{r2_v:.4f}"]
]

table = ax.table(cellText=data_table, colLabels=None, cellLoc='center', loc='center')
table.auto_set_font_size(False); table.set_fontsize(12); table.scale(1, 1.8)

# Estilo Verde
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor(COLOR_XGB) # Cabecera Verde
    else:
        cell.set_facecolor('#f8f9fa')
        cell.set_edgecolor('#dddddd')

plt.title('Resumen de Desempeño XGBoost (Downlink)', fontsize=14, fontweight='bold', y=0.95)
plt.savefig(os.path.join(DIR_OUTPUT, '7_Tabla_Metricas_Finales_DL.png'), dpi=300, bbox_inches='tight')
plt.close()

# Guardar modelo
joblib.dump(xgb_model, os.path.join(DIR_OUTPUT, 'Modelo_XGBoost_Downlink.pkl'))

print(f"\n✅ Procesamiento terminado. Gráficas y Tabla guardadas en: {DIR_OUTPUT}")